### 연습
- ratings_train.txt 파일을 로드
- 결측치 제거
- document 컬럼의 문자 정규화(특수문자 제거, 2칸 이상의 공백 제거, 좌우 공백 제거)
- 중복 document 제거 , 글자의 수가 1개 이하인 행은 제거
- DataFrame에서 sample(n = 10000, random_state=42)로 임의의 데이터를 추출하여 저장 (head() -> 상위 데이터 | tail() -> 하위 데이터 | sample() -> 무작위 데이터)
- train, test 셋으로 8:2 로 데이터분할
- sbert 모델은 'BM-K/KoSimCSE-roberta-multitask'을 이용
- Dataset을 정의 (Trainer 이용하지 않고 Dataset과 DataLoader 사용)
    - 입력받은 document와 label를 document는 SBERT 모델을 이용하여 인코딩
    - label 데이터를 tensor형태로 변환
    - __len__ 함수는 라벨의 길이를 되돌려준다
    - __getitem__ 함수는 인코딩된 데이터[idx], label[idx]를 되돌려준다
- Dataset를 train, test를 이용해서 Dataset을 생성
- DataLoarder를 이용하여 배치의 사이즈는 128 shuffle은 True 구성한다.

In [2]:
import re
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# 데이터 로드
df = pd.read_csv('../data/ratings_train.txt', sep='\t')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [4]:
# 결측치를 제거
df.dropna(subset='document', inplace=True)

In [5]:
def normalize(text):
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [6]:
df['document'] = df['document'].map(normalize)

In [7]:
# 중복 데이터를 제거
df.drop_duplicates(subset='document', inplace=True)

In [8]:
len(df)

144734

In [9]:
# 문자열의 길이가 1 이하는 제거 -> 1 초과인 데이터만 확인
flag = df['document'].str.len() > 1
df = df.loc[flag, ]

In [10]:
# 랜덤한 데이터 10000개 추출 sample()
df = df.sample(n = 10000, random_state=42).reset_index(drop=True)

In [11]:
# train, test 분할 (8:2)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

In [12]:
train_df['label'].value_counts()

label
0    4006
1    3994
Name: count, dtype: int64

In [13]:
model_name = 'BM-K/KoSimCSE-roberta-multitask'
sbert = SentenceTransformer(model_name)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [14]:
# Dataset 정의
class SBERTDataset(Dataset):
    # 생성자 함수 -> document, labels 받아와서 document 임베딩, labels는 tensor화
    def __init__(self, document, labels):
        # no_grad() -> 자동 미분 일시 정지
        # inference_mode() -> 추론 모드
        with torch.inference_mode():
            # 로드한 모델을 이용해서 encode 작업
            # convert_to_tensor -> 결과값을 tensor로 받을것인가? (False : list)
            # normalize_embeddings -> L2 정규화 할것인가?
            self.emb = sbert.encode(document, convert_to_tensor=True, normalize_embeddings=True)
        # labels를 tensor화
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        # labels의 길이를 되돌려준다
        return len(self.labels)
    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]

In [ ]:
# Dataset의 형태로 데이터프레임을 변환
train_ds = SBERTDataset(train_df['document'].tolist(), train_df['label'].tolist())
test_ds = SBERTDataset(train_df['document'].tolist(), test_df['label'].tolist())

In [ ]:
# DataLoader를 이용해서 배치 사이즈만큼의 데이터를 생성
train_dl = DataLoader(train_ds, batch_size = 128, shuffle = True)
test_dl = DataLoader(test_ds, batch_size = 128, shuffle = True)